# Project 2: Multi-Factor Quantamental Model (Semiconductors)
**Universe:** NVDA, AMD, INTC, QCOM, AVGO, TXN, MU

Builds Value, Quality, and Momentum factors, standardizes them via z-scoring, and combines them into a single ranked score — then runs a look-ahead-bias-free backtest on the Momentum factor specifically.

In [1]:
import pandas as pd
import yfinance as yf

TICKERS = ["NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN", "MU"]

## Step 1: Pull fundamentals
**Limitation, stated upfront:** these are *current* snapshot fundamentals (P/E, ROE, P/B), not point-in-time historical data — free sources like yfinance don't provide historical fundamentals. This matters for backtesting honesty (see Momentum backtest below, which avoids this issue by using only price data, which *is* properly time-stamped).

In [2]:
fundamentals = {}
for ticker in TICKERS:
    info = yf.Ticker(ticker).info
    fundamentals[ticker] = {
        "pe": info.get("trailingPE"),
        "roe": info.get("returnOnEquity"),
        "pb": info.get("priceToBook"),
    }

fundamentals_df = pd.DataFrame(fundamentals).T
fundamentals_df

,pe,roe,pb
NVDA,29.170820,1.17211,24.670464
AMD,156.135470,0.10196,15.389525
INTC,NaN,-0.10715,6.874244
QCOM,20.795275,0.33754,7.065005
AVGO,43.844444,0.44245,17.006992
TXN,42.765648,0.35176,14.896314
MU,14.771060,0.88260,12.048311


## Step 2: Momentum factor (from real price history)

In [3]:
price_data = yf.download(TICKERS, start="2026-01-01", end="2026-09-26")
closes = price_data["Close"]
momentum = (closes.iloc[-1] - closes.iloc[0]) / closes.iloc[0] * 100
momentum.sort_values(ascending=False)

/tmp/ipykernel_3145/421590441.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(TICKERS, start="2026-01-01", end="2026-09-26")
[*********************100%***********************]  7 of 7 completed


,0
Ticker,
MU,243.320359
INTC,212.341282
AMD,182.198954
TXN,59.276409
NVDA,19.458129
QCOM,18.584735
AVGO,2.052377


## Step 3: Z-score each factor
A z-score standardizes different scales (P/E in the 20s-150s vs. ROE as a small decimal) so they can be meaningfully combined. Sign convention matters: for P/E, *lower* is better (cheap), so that z-score is inverted; for ROE and Momentum, *higher* is already better.

In [4]:
def zscore(series, invert=False):
    z = (series - series.mean()) / series.std()
    return -z if invert else z

fundamentals_df["pe_zscore"] = zscore(fundamentals_df["pe"], invert=True)
fundamentals_df["roe_zscore"] = zscore(fundamentals_df["roe"], invert=False)
fundamentals_df["momentum_zscore"] = zscore(momentum, invert=False)

fundamentals_df["combined_score"] = (
    fundamentals_df["pe_zscore"] + fundamentals_df["roe_zscore"] + fundamentals_df["momentum_zscore"]
) / 3

fundamentals_df.sort_values("combined_score", ascending=False)

,pe,roe,pb,pe_zscore,roe_zscore,momentum_zscore,combined_score
MU,14.771060,0.88260,12.048311,0.692469,0.972716,1.335281,1.000155
NVDA,29.170820,1.17211,24.670464,0.419101,1.630481,-0.830775,0.406269
TXN,42.765648,0.35176,14.896314,0.161014,-0.233350,-0.445500,-0.172612
QCOM,20.795275,0.33754,7.065005,0.578104,-0.265658,-0.839226,-0.175593
AVGO,43.844444,0.44245,17.006992,0.140534,-0.027303,-0.999191,-0.295320
AMD,156.135470,0.10196,15.389525,-1.991221,-0.800894,0.743880,-0.682745
INTC,NaN,-0.10715,6.874244,NaN,-1.275991,1.035532,NaN


**Finding:** NVDA ranks #1, driven mainly by exceptional ROE. **QCOM** is the most interesting case — it screens reasonably well on Value and Quality alone, but drops sharply in the combined ranking once Momentum is added, because its recent price performance has been weak. This illustrates a real, common tension: "fundamentally cheap" and "currently in favor" are not the same thing, and a multi-factor model is built specifically to surface that divergence rather than ignore it.

## Step 4: Momentum backtest (look-ahead-bias-free)
Formation period (Jan–Apr 2026) is used to rank stocks by momentum; the test period (May–Sep 2026, non-overlapping) checks what actually happened next — simulating standing in early May with only Jan–Apr information available.

In [5]:
full_data = yf.download(TICKERS, start="2026-01-01", end="2026-09-26")
full_closes = full_data["Close"]

formation = full_closes["2026-01-01":"2026-04-30"]
test = full_closes["2026-05-01":"2026-09-26"]

formation_returns = (formation.iloc[-1] - formation.iloc[0]) / formation.iloc[0] * 100
test_returns = (test.iloc[-1] - test.iloc[0]) / test.iloc[0] * 100

comparison = pd.DataFrame({
    "formation_momentum": formation_returns,
    "test_period_return": test_returns
})
comparison.sort_values("formation_momentum", ascending=False)

/tmp/ipykernel_3145/1498276515.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  full_data = yf.download(TICKERS, start="2026-01-01", end="2026-09-26")
[*********************100%***********************]  7 of 7 completed


,formation_momentum,test_period_return
Ticker,,
INTC,139.918742,23.469179
MU,64.028014,99.636014
TXN,59.370593,-0.037761
AMD,58.629788,74.912628
AVGO,20.334175,-15.967560
NVDA,5.682182,13.673193
QCOM,4.482016,15.145613


**Finding — an honest null result:** momentum did NOT show a consistent persistence pattern in this small, 7-stock universe. The stock with the strongest formation-period momentum (INTC, +140%) did not have a correspondingly strong test-period return.

**Why:** INTC's extreme formation-period return traces to a specific, verifiable catalyst — the U.S. government's 2025 CHIPS Act equity stake in Intel, plus a genuine foundry turnaround (18A chip progress) — not organic price momentum. Once that one-off catalyst was priced in, there was no structural reason for it to repeat at the same magnitude. This is a useful illustration of why momentum strategies benefit from identifying and handling event-driven outliers separately from organic trend-following, and why a small sample is vulnerable to one stock's idiosyncratic news dominating the result.

## Summary
- Built a complete multi-factor pipeline: raw fundamentals → sign-corrected z-scores → combined ranking, with an explainable divergence (QCOM) surfaced by the model.
- Ran an honest, methodologically sound backtest (proper formation/test split) rather than only reporting favorable results.
- Investigated and sourced the dataset's biggest outlier rather than leaving it unexplained.
- Explicitly stated the project's real data limitation (no point-in-time fundamentals from free sources) rather than glossing over it.
